# Jour 2 - Nettoyage des données TMDB
Exploration pas à pas. Le code réutilisable est dans `src/cleaning/clean_data.py`.

In [3]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.cleaning.clean_data import (load_raw_movies, missing_summary, detect_inconsistencies,
                                     clean_movies, split_variable_types, save_clean)
pd.set_option('display.max_columns', 30)

## 1. Chargement : JSON imbriqués → DataFrame plat

In [4]:
raw = load_raw_movies()
print(raw.shape)
raw.head()

(2000, 14)


,movie_id,title,overview,release_date,runtime,original_language,status,genres,keywords,budget,revenue,popularity,vote_average,vote_count
0,100042,Dumb and Dumber To,"20 years since their first adventure, Lloyd an...",2014-11-12,110,en,Released,[Comedy],"[friendship, road trip, sequel, buddy comedy, ...",40000000,169837010,13.6220,5.600,3516
1,1000866,The Marching Band,A world-famous orchestra conductor discovers t...,2024-11-27,102,fr,Released,"[Comedy, Drama, Music]","[brotherhood, marching band, adopted, orchestr...",6100000,26252618,21.5722,7.300,523
2,10009,Brother Bear,When an impulsive boy named Kenai is magically...,2003-10-23,85,en,Released,"[Adventure, Animation, Family]","[friendship, sibling relationship, transformat...",128000000,250397798,18.9477,7.285,5874
3,10010,Brother Bear 2,Kenai finds his childhood human friend Nita an...,2006-07-19,73,en,Released,"[Adventure, Animation, Family]","[grizzly bear, human animal relationship, fore...",0,0,12.2858,6.700,1596
4,10020,Beauty and the Beast,"Follow the adventures of Belle, a bright young...",1991-10-22,84,en,Released,"[Romance, Family, Animation, Fantasy]","[princess, france, prince, castle, villain, ro...",25000000,424967620,24.2643,7.723,10912


## 2. Types et structure

In [5]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   movie_id           2000 non-null   int64  
 1   title              2000 non-null   str    
 2   overview           2000 non-null   str    
 3   release_date       2000 non-null   str    
 4   runtime            2000 non-null   int64  
 5   original_language  2000 non-null   str    
 6   status             2000 non-null   str    
 7   genres             2000 non-null   object 
 8   keywords           2000 non-null   object 
 9   budget             2000 non-null   int64  
 10  revenue            2000 non-null   int64  
 11  popularity         2000 non-null   float64
 12  vote_average       2000 non-null   float64
 13  vote_count         2000 non-null   int64  
dtypes: float64(2), int64(5), object(2), str(5)
memory usage: 218.9+ KB


In [6]:
raw[['runtime','budget','revenue','popularity','vote_average','vote_count']].describe().round(2).T

,count,mean,std,min,25%,50%,75%,max
runtime,2000.0,1.139000e+02,2.343000e+01,0.00,99.00,1.100000e+02,1.260000e+02,2.540000e+02
budget,2000.0,6.000923e+07,6.789350e+07,0.00,10000000.00,3.500000e+07,9.000000e+07,6.588000e+08
revenue,2000.0,2.182950e+08,3.069443e+08,0.00,22816572.00,1.156239e+08,2.900791e+08,2.923706e+09
popularity,2000.0,2.274000e+01,2.868000e+01,8.78,12.96,1.603000e+01,2.262000e+01,6.462600e+02
vote_average,2000.0,6.930000e+00,7.900000e-01,3.80,6.41,6.950000e+00,7.460000e+00,9.160000e+00
vote_count,2000.0,5.770520e+03,5.859650e+03,10.00,1734.75,4.119500e+03,7.712000e+03,4.130700e+04


## 3. Valeurs manquantes et doublons

In [7]:
missing_summary(raw)

,manquants,pourcentage
movie_id,0,0.0
title,0,0.0
overview,0,0.0
release_date,0,0.0
runtime,0,0.0
original_language,0,0.0
status,0,0.0
genres,0,0.0
keywords,36,1.8
budget,0,0.0


In [8]:
print('Doublons movie_id :', raw['movie_id'].duplicated().sum())
print('Doublons (title, release_date) :', raw.duplicated(subset=['title','release_date']).sum())

Doublons movie_id : 0
Doublons (title, release_date) : 0


## 4. Incohérences
`budget = 0` et `revenue = 0` ne sont pas de vrais zéros : c'est la valeur que TMDB renvoie quand l'information est inconnue.

In [11]:
pd.Series(detect_inconsistencies(raw), name='nombre').to_frame()

,nombre
movie_id dupliqués,0
"(title, release_date) dupliqués",0
budget = 0 (valeur bidon),282
budget entre 1 et 999 (erreur de saisie),1
revenue = 0 (valeur bidon),267
revenue entre 1 et 999 (erreur de saisie),2
runtime = 0 ou négatif,3
runtime > 300 min (suspect),0
vote_count = 0 mais vote_average > 0,0
"vote_average hors [0, 10]",0


## 5. Nettoyage
Aucune imputation ici (risque de Data Leakage) : les valeurs inconnues restent `NaN`.

In [12]:
clean, log = clean_movies(raw)
pd.Series(log, name='valeur').to_frame()

,valeur
lignes sans id/titre supprimées,0
doublons movie_id supprimés,0
"doublons (title, release_date) supprimés",0
budget mis à NaN (< 1000),283
revenue mis à NaN (< 1000),269
runtime mis à NaN (<= 0),3
vote_average mis à NaN (0 vote),0
films non 'Released' supprimés,0
lignes finales,2000


In [13]:
clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   movie_id           2000 non-null   int64         
 1   title              2000 non-null   object        
 2   overview           2000 non-null   object        
 3   release_date       2000 non-null   datetime64[us]
 4   runtime            1997 non-null   float64       
 5   original_language  2000 non-null   object        
 6   status             2000 non-null   object        
 7   genres             2000 non-null   object        
 8   keywords           2000 non-null   object        
 9   budget             1717 non-null   float64       
 10  revenue            1731 non-null   float64       
 11  popularity         2000 non-null   float64       
 12  vote_average       2000 non-null   float64       
 13  vote_count         2000 non-null   int64         
dtypes: datetime64[us](1

In [14]:
missing_summary(clean)

,manquants,pourcentage
movie_id,0,0.0
title,0,0.0
overview,0,0.0
release_date,0,0.0
runtime,3,0.2
original_language,0,0.0
status,0,0.0
genres,0,0.0
keywords,36,1.8
budget,283,14.2


## 6. Séparation des variables

In [15]:
types = split_variable_types(clean)
types

{'id': ['movie_id'],
 'numeric': ['runtime',
  'budget',
  'revenue',
  'popularity',
  'vote_average',
  'vote_count'],
 'categorical': ['original_language', 'status'],
 'text': ['title', 'overview'],
 'date': ['release_date'],
 'list': ['genres', 'keywords']}

## 7. Sauvegarde
(le script `python src/cleaning/clean_data.py` fait la même chose et écrit le rapport dans `reports/`)

In [16]:
save_clean(clean)
clean.head()

,movie_id,title,overview,release_date,runtime,original_language,status,genres,keywords,budget,revenue,popularity,vote_average,vote_count
0,11,Star Wars,Princess Leia is captured and held hostage by ...,1977-05-25,121.0,en,Released,"[Adventure, Action, Science Fiction]","[empire, galaxy, rebellion, android, hermit, s...",11000000.0,775398007.0,34.3206,8.207,22927
1,12,Finding Nemo,"Nemo, an adventurous young clownfish, is unexp...",2003-05-30,100.0,en,Released,"[Animation, Family, Adventure]","[fish, sydney, australia, parent child relatio...",94000000.0,940335536.0,37.4962,7.822,21029
2,13,Forrest Gump,A man with a low IQ has accomplished great thi...,1994-06-23,142.0,en,Released,"[Comedy, Drama, Romance]","[new year's eve, vietnam war, vietnam veteran,...",55000000.0,677387716.0,36.0453,8.461,30542
3,14,American Beauty,"Lester Burnham, a depressed suburban father in...",1999-09-15,122.0,en,Released,[Drama],"[estate agent, adultery, coming out, first tim...",15000000.0,356296601.0,19.7344,7.997,13393
4,15,Citizen Kane,Newspaper magnate Charles Foster Kane is taken...,1941-04-17,119.0,en,Released,"[Mystery, Drama]","[media tycoon, florida, art collector, newspap...",839727.0,23218000.0,11.3980,7.968,6248
